Assignment 4: Forward–Backward Exponential Smoothing and Its Advantages Over Running Means

Group 6: Pavel Kuznetsov, Olga Tiupina, Sofya Konstantinova, Skoltech, 2025

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import math
from mpl_toolkits.mplot3d import Axes3D


def smooth_data(dates, values):
    """
    Smoothing data function
    
    param:
    dates - timeline array
    values - an array of values corresponding to dates
    """
    temp_df = pd.DataFrame({'date': dates, 'value': values})
    temp_df = temp_df.set_index('date')    

    # create cope of values
    data = temp_df['value'].values.copy()
    n = len(data)

    smoothed_data = np.zeros(n)
    
    # find average for first and last 6 data points 
    # first
    average_first_6 = np.mean(data[:6])
    smoothed_data[:6] = average_first_6
    # end
    average_last_6 = np.mean(data[-6:])
    smoothed_data[-6:] = average_last_6

    # smoothing process for point from 5 до n-6 by 13-month running mean
    for t in range(5, n-6):
        smoothed_data[t] = (1/24 * data[t-6] + 
                   1/24 * data[t+6] + 
                   1/12 * (data[t] + data[t+1] + data[t+2] + data[t+3] + 
                           data[t+4] + data[t+5] + data[t-1] + data[t-2] + 
                           data[t-3] + data[t-4] + data[t-5]))

    temp_df['value_smooth'] = smoothed_data
    
    return temp_df


def deviation_coefficient(x, z):
    sum = 0
    for i in range(0, len(x)):
        sum += ((x[i] - z[i])**2)
    return sum

def variability_coefficient(x):
    sum = 0
    for i in range(0, len(x) - 2):
        sum += ((x[i + 2] - 2*x[i + 1] + x[i])**2)
    return sum

# Data loading
def load_surface_data(filename):
    """Loads surface data from a text file"""
    data = []
    with open(filename, 'r') as file:
        for line in file:
            row = line.strip().split()
            row_data = [float(x) for x in row if x]
            if row_data: 
                data.append(row_data)
    return np.array(data)


def generate_forward_exponential_smoothing(measurements, alfa):
    """
    Generate forward exponential smoothing
    """
    num = len(measurements)
    X_ex = np.zeros(num)
    X_ex[0] = measurements[0]
    
    # exponential smoothing
    for i in range(1, num):
        X_ex[i] = alfa * measurements[i] + (1 - alfa) * X_ex[i-1] 
    
    return X_ex

def apply_forward_exponential_smoothing_to_rows(surface, alpha):
    """
    Generate forward exponential smoothing for rows
    """
    rows, cols = surface.shape
    smoothed_surface = np.zeros((rows, cols))
    
    for i in range(rows):
        smoothed_surface[i] = generate_forward_exponential_smoothing(surface[i], alpha)
    
    return smoothed_surface

def apply_forward_exponential_smoothing_to_columns(surface, alpha):
    """
    Generate forward exponential smoothing for columns
    """
    rows, cols = surface.shape
    smoothed_surface = np.zeros((rows, cols))
    
    for j in range(cols):
        smoothed_surface[:, j] = generate_forward_exponential_smoothing(surface[:, j], alpha)
    
    return smoothed_surface

def generate_backward_exponential_smoothing(X_f, alfa):
    
    """
    Generate forward exponential smoothing
    """
    
    num = len(X_f)
    X_b = np.zeros(num)
    X_b[-1] = X_f[-1]

    # backward exponential smoothing
    for i in range(num - 2, -1, -1):
        X_b[i] = X_b[i + 1] + alfa * (X_f[i] - X_b[i + 1])
    
    return X_b

def apply_backward_exponential_smoothing_to_rows(surface, alpha):
    """
    Generate backward exponential smoothing for rows
    """
    rows, cols = surface.shape
    smoothed_surface = np.zeros((rows, cols))
    
    for i in range(rows):
        smoothed_surface[i] = generate_backward_exponential_smoothing(surface[i], alpha)
    
    return smoothed_surface

def apply_backward_exponential_smoothing_to_columns(surface, alpha):
    """
    Generate backward exponential smoothing for columns
    """
    rows, cols = surface.shape
    smoothed_surface = np.zeros((rows, cols))
    
    for j in range(cols):
        smoothed_surface[:, j] = generate_backward_exponential_smoothing(surface[:, j], alpha)
    
    return smoothed_surface

In [ ]:
fontsize_title = 30
fontsize_label = 20
fontsize_leg = 18

In [ ]:
# read data file
df1 = pd.read_csv('../ass4/data/data_group6.txt',sep=r'\s+', names=['year', 'month', 'value'], header=None) 
df1['date'] = pd.to_datetime(df1['year'].astype(str) + '-' + df1['month'].astype(str))

We ploted monthly mean sunspot number charts real from a file data_group6.txt and smoothed it out using a 13-month running mean

In [ ]:
# smooth both datasets
smoothed_sunspots = smooth_data(df1['date'], df1['value'])

plt.figure(figsize=(20, 8))

plt.plot(df1['date'], df1["value"], 'gray',linewidth=1, alpha=0.5, label='Original sunspots')
plt.plot(smoothed_sunspots.index, smoothed_sunspots['value_smooth'], 'blue',linewidth=1.5, label='Smoothed sunspots')
plt.xlabel('\n Data, year \n', fontsize=fontsize_label)
plt.ylabel('\n Monthly mean sunspot number \n', fontsize=fontsize_label)
plt.title('\n Fig.1. Sunspot Data: Original vs Smoothed \n', fontsize=fontsize_title)
plt.legend(fontsize=fontsize_leg)

plt.grid(True, alpha=0.3)
plt.tick_params(axis='x', rotation=45)

We calculated a smoothing constant (𝛼) based on deviation and variability indicators

In [ ]:
# find optimal alpha  by metric
measurements = df1['value'].values
smoothing_acc_trajectory_ES = []  
deviation_coefficients = []       
variability_coefficients = []    
alphas = np.arange(0.1, 1.0, 0.01)

for i in range(0, len(alphas)):
    try:
        smoothed_data_2 =  generate_forward_exponential_smoothing(measurements, alphas[i])
        smoothed_data =  generate_backward_exponential_smoothing(smoothed_data_2, alphas[i])
        smoothing_acc_trajectory_ES.append(smoothed_data)
        original_cut = measurements[:len(smoothed_data)]
        
        dev_coef = deviation_coefficient(smoothed_data, original_cut)
        var_coef = variability_coefficient(smoothed_data)
        
        deviation_coefficients.append(dev_coef)
        variability_coefficients.append(var_coef)
        
    except Exception as e:
        print(f"Error for alpha={i}: {e}")
        continue

deviation_coefficients = np.array(deviation_coefficients)
variability_coefficients = np.array(variability_coefficients)

To compare methods of smoothing, we calculated Deviation and Variability indicators for 13-month Running Mean

In [ ]:
dev_coef_running = deviation_coefficient(measurements, np.array(smoothed_sunspots['value_smooth'].values))

print(f"Deviation indicator for Running Mean: {dev_coef_running}")


var_coef_running = variability_coefficient(np.array(smoothed_sunspots['value_smooth'].values))

print(f"Variability indicator for Running Mean: {var_coef_running}")

In [ ]:
plt.figure(figsize=(20, 10))

plt.axhline(y=dev_coef_running, color="blue", linestyle="--", linewidth=2, alpha=0.7, label=f'Deviation Coefficient RM')
plt.axhline(y=var_coef_running, color="red", linestyle="--", linewidth=2, alpha=0.7, label=f'Variability Coefficient RM')
plt.plot(alphas, deviation_coefficients, color="blue", linewidth=2, alpha=0.8, label='Deviation Coefficient ES')
plt.plot(alphas, variability_coefficients, color="red", linewidth=2, alpha=0.8, label='Variability Coefficient ES')
plt.title('\n Fig.2. Optimal Smoothing Coefficient Selection \n', fontsize=fontsize_title)
plt.xlabel('Alpha', fontsize=fontsize_label)
plt.ylabel('Coefficient Value', fontsize=fontsize_label)
plt.grid(True, alpha=0.3)


plt.legend(fontsize=fontsize_leg)
plt.show()

In [ ]:
plt.figure(figsize=(20, 10))

plt.axhline(y=dev_coef_running, color="blue", linestyle="--", linewidth=2, alpha=0.7, label=f'Deviation Coefficient RM')
plt.axhline(y=var_coef_running, color="red", linestyle="--", linewidth=2, alpha=0.7, label=f'Variability Coefficient RM')
plt.plot(alphas, deviation_coefficients, color="blue", linewidth=2, alpha=0.8, label='Deviation Coefficient ES')
plt.plot(alphas, variability_coefficients, color="red", linewidth=2, alpha=0.8, label='Variability Coefficient ES')
plt.title('\n Fig.3. Optimal Smoothing Coefficient Selection \n', fontsize=fontsize_title)
plt.xlabel('Alpha', fontsize=fontsize_label)
plt.ylabel('Coefficient Value', fontsize=fontsize_label)
plt.grid(True, alpha=0.3)

plt.xlim(0.2, 0.3)
plt.ylim(0, 2000)

plt.legend(fontsize=fontsize_leg)
plt.show()

In [ ]:
plt.figure(figsize=(20, 10))

plt.axhline(y=dev_coef_running, color="blue", linestyle="--", linewidth=2, alpha=0.7, label=f'Deviation Coefficient RM')
plt.axhline(y=var_coef_running, color="red", linestyle="--", linewidth=2, alpha=0.7, label=f'Variability Coefficient RM')
plt.plot(alphas, deviation_coefficients, color="blue", linewidth=2, alpha=0.8, label='Deviation Coefficient ES')
plt.plot(alphas, variability_coefficients, color="red", linewidth=2, alpha=0.8, label='Variability Coefficient ES')
plt.title('\n Fig.4. Optimal Smoothing Coefficient Selection \n', fontsize=fontsize_title)
plt.xlabel('Alpha', fontsize=fontsize_label)
plt.ylabel('Coefficient Value', fontsize=fontsize_label)
plt.grid(True, alpha=0.3)

plt.xlim(0.2, 0.3)
plt.ylim(140000, 200000)

plt.legend(fontsize=fontsize_leg)
plt.show()

We found intersection point alpha  of 2 graphs - deviation coefficient and variability coefficient. Also it below the RM Deviation Coefficient, so it is is better

In [ ]:
intersection_idx = np.argmin(np.abs(deviation_coefficients - variability_coefficients))
intersection_alpha = alphas[intersection_idx]

print(f"Intersection point: alpha = {intersection_alpha:.3f}")

In [ ]:
measurements = df1['value'].values
forward = generate_forward_exponential_smoothing(measurements=measurements ,alfa=0.43)
backward = generate_backward_exponential_smoothing(X_f = forward, alfa = 0.43)

In [ ]:
df1['forward'] = forward
df1['backward'] = backward

Using the found alpha, we built Forward and Backward Smoothed sunspot number graphs

In [ ]:
plt.figure(figsize=(20, 10))

plt.plot(df1['date'], df1['value'], color = 'black', linestyle='--', alpha=0.7, linewidth=1, label='Original sunspots')
plt.plot(df1['date'], df1['forward'], color ='green', linewidth=1, label='Forward exponential smoothing')
plt.plot(df1['date'], df1['backward'], color ='red', linewidth=2, label='Backward exponential smoothing')

plt.xlabel('Data, year', fontsize=fontsize_label)
plt.ylabel('Monthly mean sunspot number', fontsize=12)
plt.title('\n Fig.5. Sunspot Data: Original, Forward and Backward Smoothed \n', fontsize=fontsize_title)
plt.legend(fontsize=fontsize_leg)
plt.grid(True, alpha=0.3)
plt.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show() 

In [ ]:
plt.figure(figsize=(20, 10))

plt.plot(df1['date'], df1['value'], color = 'black', linestyle='--', alpha=0.7, linewidth=1, label='Original sunspots')
plt.plot(df1['date'], df1['forward'], color = "green", linewidth=1, label='Forward exponential smoothing')
plt.plot(df1['date'], df1['backward'], color ='red', linewidth=2, label='Backward exponential smoothing')

plt.xlabel('Data, year', fontsize=fontsize_label)
plt.ylabel('Monthly mean sunspot number', fontsize=12)
plt.title('\n Fig.6. Sunspot Data: Original, Forward and Backward Smoothed with zooming \n', fontsize=fontsize_title)

plt.xlim(pd.Timestamp('1836-01-01'), pd.Timestamp('1844-12-31'))
plt.legend(fontsize=fontsize_leg)
plt.grid(True, alpha=0.3)
plt.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show() 

In [ ]:
plt.figure(figsize=(20, 10))

plt.plot(df1['date'], df1['value'], color = 'black', linestyle='--', alpha=0.7, linewidth=1, label='Original sunspots')
plt.plot(df1['date'], df1['backward'], color ='red', linewidth=2, label='Backward exponential smoothing')
plt.plot(smoothed_sunspots.index, smoothed_sunspots['value_smooth'], 'blue',linewidth=1.5, label='Running mean, M = 13')
plt.xlabel('Data, year', fontsize=fontsize_label)
plt.ylabel('Monthly mean sunspot number', fontsize=fontsize_label)
plt.title('\n Fig.7. Sunspot Data: Original, Running mean and Backward Smoothed \n', fontsize=fontsize_title)

plt.legend(fontsize=fontsize_leg)
plt.grid(True, alpha=0.3)
plt.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show() 

Applying the forward-backward exponential smoothing gives a better estimate

In [ ]:
plt.figure(figsize=(20, 10))

plt.plot(df1['date'], df1['value'], color = 'black', linestyle='--', alpha=0.7, linewidth=1, label='Original sunspots')
plt.plot(df1['date'], df1['backward'], color ='red', linewidth=2, label='Backward exponential smoothing')
plt.plot(smoothed_sunspots.index, smoothed_sunspots['value_smooth'], 'blue',linewidth=1.5, label='Running mean, M = 13')
plt.xlabel('Data, year', fontsize=fontsize_label)
plt.ylabel('Monthly mean sunspot number', fontsize=fontsize_label)
plt.title('\n Fig.8. Sunspot Data: Original, Running mean and Backward Smoothed \n', fontsize=fontsize_title)


plt.xlim(pd.Timestamp('1836-01-01'), pd.Timestamp('1844-12-31'))
plt.legend(fontsize=fontsize_leg)
plt.grid(True, alpha=0.3)
plt.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show() 

We compared Running mean with M = 13 and Forward-Backward Smoothing. Forward-Backward Smoothing has better results

To make it easier to compare the difference between the noisy surface and the true surface, let's plot them

In [ ]:
try:
    noisy_surface = load_surface_data('data/noisy_surface.txt')
    true_surface = load_surface_data('data/true_surface.txt')
    
except FileNotFoundError as e:
    exit()

fig = plt.figure(figsize=(18, 8))

ax1 = fig.add_subplot(121, projection='3d')
x = np.arange(noisy_surface.shape[1])
y = np.arange(noisy_surface.shape[0])
X, Y = np.meshgrid(x, y)

surf1 = ax1.plot_surface(X, Y, noisy_surface, cmap='plasma', 
                        alpha=0.9, linewidth=0, antialiased=True)
ax1.set_title('\n Fig.9. Noisy Surface 3D \n', fontsize=fontsize_title, fontweight='bold')
ax1.set_xlabel('\n X Coordinate \n', fontsize=fontsize_label)
ax1.set_ylabel('\n Y Coordinate \n', fontsize=fontsize_label)
ax1.set_zlabel('\n Value \n', fontsize=fontsize_label)
fig.colorbar(surf1, ax=ax1, shrink=0.5, aspect=20)

ax2 = fig.add_subplot(122, projection='3d')
surf2 = ax2.plot_surface(X, Y, true_surface, cmap='plasma', 
                        alpha=0.9, linewidth=0, antialiased=True)
ax2.set_title('\n Fig.10. True Surface 3D \n', fontsize=fontsize_title, fontweight='bold')
ax2.set_xlabel('\n X Coordinate \n', fontsize=fontsize_label)
ax2.set_ylabel('\n Y Coordinate \n', fontsize=fontsize_label)
ax2.set_zlabel('\n Value \n', fontsize=fontsize_label)
fig.colorbar(surf2, ax=ax2, shrink=0.5, aspect=20)

ax1.view_init(elev=30, azim=45)
ax2.view_init(elev=30, azim=45)

plt.tight_layout()
plt.show()


In [ ]:
# difference between surfaces
deviation = noisy_surface - true_surface

# reshape matrix into a single array 
deviation_flat = deviation.reshape(-1)

variance = 0
for i in range(0, len(deviation_flat)):
    variance += deviation_flat[i]**2 
variance /= len(deviation_flat)

print(f'variance between noisy surface & true surface: {variance}')

In [ ]:
alpha = 0.335

step1 = apply_forward_exponential_smoothing_to_rows(noisy_surface, alpha)
step2 = apply_backward_exponential_smoothing_to_rows(step1, alpha)
step3 = apply_forward_exponential_smoothing_to_columns(step2, alpha)
smoothed_surface = apply_backward_exponential_smoothing_to_columns(step3, alpha)

fig = plt.figure(figsize=(20, 7))

x = np.arange(noisy_surface.shape[1])
y = np.arange(noisy_surface.shape[0])
X, Y = np.meshgrid(x, y)

ax1 = fig.add_subplot(131, projection='3d')
ax2 = fig.add_subplot(132, projection='3d')
ax3 = fig.add_subplot(133, projection='3d')

surf1 = ax1.plot_surface(X, Y, noisy_surface, cmap='plasma', alpha=0.9, linewidth=0, antialiased=True)
surf2 = ax2.plot_surface(X, Y, smoothed_surface, cmap='plasma', alpha=0.9, linewidth=0, antialiased=True)
surf3 = ax3.plot_surface(X, Y, true_surface, cmap='plasma', alpha=0.9, linewidth=0, antialiased=True)

titles = ['\n Fig.11. Noisy \n Surface 3D \n', '\n Fig.12. Smoothed \n Surface 3D \n', '\n Fig.13. True \n Surface 3D \n']
for ax, title in zip([ax1, ax2, ax3], titles):
    ax.set_title(title, fontsize=fontsize_title, fontweight='bold')
    ax.set_xlabel('\n X Coordinate \n', fontsize=fontsize_label)
    ax.set_ylabel('\n Y Coordinate \n', fontsize=fontsize_label)
    ax.set_zlabel('\n Value \n', fontsize=fontsize_label)
    ax.view_init(elev=30, azim=45)

cbar_ax = fig.add_axes([0.15, 0.05, 0.7, 0.03])
fig.colorbar(surf1, cax=cbar_ax, orientation='horizontal', label='Value')

plt.show()

Visually comparing the obtained evaluation results with the true surface, we can conclude that the use of forward-backward exponential smoothing to filter noisy surface measurements gives a good estimate

In [ ]:
# difference between surfaces
deviation = smoothed_surface - true_surface

# reshape matrix into a single array 
deviation_flat = deviation.reshape(-1)

variance = 0
for i in range(0, len(deviation_flat)):
    variance += deviation_flat[i]**2 
variance /= len(deviation_flat)

print(f'variance between smoothed surface & true surface: {variance}')

 Applying forward-backward exponential smoothing we reached smaller deviation between the smoothed surface and the true surface of 6.8 compared to deviation between the noisy surface and the true surface of 122. A lower variance in the smoothed surface indicates a better fit to the true surface, demonstrating the effectiveness of the smoothing process over using the noisy surface alone

In [ ]:
alphas = [0.1, 0.2, 0.4, 0.5, 0.7]
count = 14

for a in alphas:
    step1 = apply_forward_exponential_smoothing_to_rows(noisy_surface, a)
    step2 = apply_backward_exponential_smoothing_to_rows(step1, a)
    step3 = apply_forward_exponential_smoothing_to_columns(step2, a)
    smoothed_surface = apply_backward_exponential_smoothing_to_columns(step3, a)

    fig = plt.figure(figsize=(20, 7))

    x = np.arange(noisy_surface.shape[1])
    y = np.arange(noisy_surface.shape[0])
    X, Y = np.meshgrid(x, y)

    ax1 = fig.add_subplot(121, projection='3d')
    ax2 = fig.add_subplot(122, projection='3d')

    surf1 = ax1.plot_surface(X, Y, smoothed_surface, cmap='plasma', alpha=0.9, linewidth=0, antialiased=True)
    surf2 = ax2.plot_surface(X, Y, true_surface, cmap='plasma', alpha=0.9, linewidth=0, antialiased=True)
    titles = [f'\n Fig.{count}. Smoothed Surface 3D,alpha = {a} \n', f'\n Fig.{count +1}. True Surface 3D \n']
    
    for ax, title in zip([ax1, ax2], titles):
        ax.set_title(title, fontsize=fontsize_title, fontweight='bold')
        ax.set_xlabel('\n X Coordinate \n', fontsize=fontsize_label)
        ax.set_ylabel('\n Y Coordinate \n', fontsize=fontsize_label)
        ax.set_zlabel('\n Value \n', fontsize=fontsize_label)
        ax.view_init(elev=30, azim=45)

    cbar_ax = fig.add_axes([0.15, 0.05, 0.7, 0.03])
    fig.colorbar(surf1, cax=cbar_ax, orientation='horizontal', label='Value')
    count += 2

    plt.show()

We have used the forward-backward smoothing for different smoothing coefficient. At low alpha, there is no noise, but the surface is very different from the true surface. At high alpha, noise is observed, which is consistent with theory, because we begin to trust noisy measurements more, but the shape of the surface corresponds to the true surface

Conclusion:

As real researchers, we engaged in noise filtering for the subsequent detection of hidden structures. We looked at how classical smoothing methods (mean and exponential smoothing) work in relation to solar activity cycles, and chose the right smoothing strategy in conditions of uncertainty using deviation and variability indicators

We have reconstructed a 3D surface using forward-backward exponential smoothing to filter the noisy surface measurements

For this we used 4 steps algorithm: 
1. Forward exponential smoothing for rows
2. Backward exponential smoothing for rows
3. Forward exponential smoothing for  columns
4. Backward exponential smoothing for  columns

After that, we used this approach for various smoothing coefficient values and drew appropriate conclusions about the dependence of the quality of the reconstructed surface depending on the smoothing coefficient values